# UTA Method — Inconsistency Resolution (Task 2.1)

This notebook implements the UTA preference disaggregation method with inconsistency resolution. Given a set of pairwise comparisons (preferential information), we find all minimal subsets of constraints that need to be removed to restore consistency.

**Decision problem**: Selecting the best European country to live in from the perspective of a student based in Poznan, Poland.

**Dataset**: OECD Better Life Index — 26 European countries evaluated on 8 criteria.

In [ ]:
import sys
import pathlib
import pandas as pd

# Add project root to path for imports
PROJECT_ROOT = pathlib.Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from common.config import GAMMA, WEIGHT_UB, WEIGHT_LB, DELTA, MIN_SEGMENT_SHARE, NON_LINEARITY_THRESHOLD
from common.data_loading import load_data, load_preferences
from common.uta_core import compute_characteristic_points
from uta_inconsistencies.resolver import find_all_minimal_removals, build_inconsistency_milp

## 1. Dataset Overview

In [11]:
df, directions = load_data()
criteria = list(directions.keys())

print(f"Alternatives: {len(df)} countries")
print(f"Criteria: {len(criteria)}")
df

Alternatives: 26 countries
Criteria: 8


,Country,Employment rate,Long-term unemployment rate,Personal earnings,Life expectancy,Life satisfaction,Employees working very long hours,Air pollution,Distance from Poznan (km)
0,Austria,72.0,1.3,53132.0,82.0,7.2,5.3,12.2,468.5
1,Belgium,65.0,2.3,54327.0,82.1,6.8,4.3,12.8,883.8
2,Czech Republic,74.0,0.6,29885.0,79.3,6.9,4.5,17.0,311.7
3,Denmark,74.0,0.9,58430.0,81.5,7.5,1.1,10.0,461.5
4,Estonia,74.0,1.2,30720.0,78.8,6.5,2.2,5.9,920.1
5,Finland,72.0,1.2,46230.0,82.1,7.9,3.6,5.5,993.3
6,France,65.0,2.9,45581.0,82.9,6.7,7.7,11.4,1098.7
7,Germany,77.0,1.2,53745.0,81.4,7.3,3.9,12.0,238.8
8,Greece,56.0,10.8,27207.0,81.7,5.8,4.5,14.5,1688.1
9,Hungary,70.0,1.2,25409.0,76.4,6.0,1.5,16.7,566.3


In [ ]:
# Criteria summary: type (gain/cost) and range
from common.config import METADATA_FILE
meta = pd.read_csv(METADATA_FILE)
summary = []
for _, row in meta.iterrows():
    c = row["criterion"]
    nature = row["nature"]
    col_min, col_max = df[c].min(), df[c].max()
    summary.append({"Criterion": c, "Nature": nature, "Min": col_min, "Max": col_max})

pd.DataFrame(summary)

## 2. Preferential Information

We define 10 pairwise comparisons as the decision maker's preferential information. These include:
- **7 consistent comparisons** based on clear advantages across multiple criteria
- **3 comparisons forming an intentional cycle**: Denmark ≻ Sweden ≻ Norway ≻ Denmark

The cycle is designed so that each individual comparison is defensible (each pair is balanced across criteria), but together they create an inconsistency — no additive value function can satisfy U(DK) > U(SE) > U(NO) > U(DK).

In [13]:
preferences = load_preferences()

pref_df = pd.DataFrame(preferences, columns=["Preferred", "Over"])
pref_df.index.name = "Index"

# Mark cycle comparisons
cycle_indices = {7, 8, 9}
pref_df["Part of cycle"] = pref_df.index.map(lambda i: "Yes" if i in cycle_indices else "No")
pref_df

,Preferred,Over,Part of cycle
Index,,,
0,Switzerland,Netherlands,No
1,Germany,Czech Republic,No
2,Norway,United Kingdom,No
3,Finland,Poland,No
4,Netherlands,Belgium,No
5,Luxembourg,Ireland,No
6,Germany,Slovenia,No
7,Denmark,Sweden,Yes
8,Sweden,Norway,Yes


### Detailed look at the Scandinavian cycle

Each comparison in the cycle is defensible depending on which criteria the DM emphasizes:

In [14]:
# Compare the three Scandinavian countries on all criteria
scandinavian = ["Denmark", "Sweden", "Norway"]
cycle_df = df[df["Country"].isin(scandinavian)].set_index("Country").loc[scandinavian]

# Add nature info
nature_row = pd.Series({c: directions[c] for c in criteria}, name="Direction (+1=gain, -1=cost)")
display(pd.concat([cycle_df[criteria], nature_row.to_frame().T]))

# Show who is better on each criterion for each pair in the cycle
cycle_pairs = [("Denmark", "Sweden"), ("Sweden", "Norway"), ("Norway", "Denmark")]
for a, b in cycle_pairs:
    a_better, b_better = [], []
    for c in criteria:
        va, vb = df.set_index("Country").loc[a, c], df.set_index("Country").loc[b, c]
        if directions[c] == 1:
            winner = a if va > vb else (b if vb > va else "tied")
        else:
            winner = a if va < vb else (b if vb < va else "tied")
        if winner == a: a_better.append(c)
        elif winner == b: b_better.append(c)
    print(f"\n{a} ≻ {b}: {a} better on {len(a_better)} criteria, {b} better on {len(b_better)}")

,Employment rate,Long-term unemployment rate,Personal earnings,Life expectancy,Life satisfaction,Employees working very long hours,Air pollution,Distance from Poznan (km)
Denmark,74.0,0.9,58430.0,81.5,7.5,1.1,10.0,461.5
Sweden,75.0,1.0,47020.0,83.2,7.3,0.9,5.8,773.1
Norway,75.0,0.9,55780.0,83.0,7.3,1.4,6.7,917.2
"Direction (+1=gain, -1=cost)",1.0,-1.0,1.0,1.0,1.0,-1.0,-1.0,-1.0



Denmark ≻ Sweden: Denmark better on 4 criteria, Sweden better on 4

Sweden ≻ Norway: Sweden better on 4 criteria, Norway better on 2

Norway ≻ Denmark: Norway better on 3 criteria, Denmark better on 4


## 3. UTA Model Setup

The UTA method constructs an additive value function with piecewise-linear marginal value functions:

$$U(a) = \sum_{i=1}^{n} u_i(g_i(a))$$

where each $u_i$ is defined by $\gamma + 1$ characteristic points. The model is subject to:

| Constraint | Description |
|---|---|
| **Normalization** | $u_i(\alpha_i) = 0$, $\sum u_i(\beta_i) = 1$ |
| **Monotonicity** | $u_i(x_i^{j+1}) \geq u_i(x_i^j)$ |
| **Weight bounds** | $0.0625 \leq u_i(\beta_i) \leq 0.5$ |
| **Anti-flatness** | Each segment $\geq 5\%$ of criterion weight |
| **Non-linearity** | First and last segment must differ by $\geq 10\%$ of weight |

For inconsistency resolution, each preference constraint is gated by a binary variable $v_{a,b}$:

$$U(a) \geq U(b) + \delta - v_{a,b}$$

- $v_{a,b} = 0$: constraint is active (preference enforced)
- $v_{a,b} = 1$: constraint is deactivated (preference removed)

**Objective**: $\min \sum v_{a,b}$ — remove as few comparisons as possible.

In [15]:
# Display characteristic points for each criterion
char_points = compute_characteristic_points(df, directions)

cp_data = []
for c in criteria:
    nature = "gain" if directions[c] == 1 else "cost"
    pts = char_points[c]
    cp_data.append({
        "Criterion": c,
        "Nature": nature,
        "x⁰ (worst)": f"{pts[0]:.1f}",
        "x¹": f"{pts[1]:.1f}",
        "x²": f"{pts[2]:.1f}",
        "x³ (best)": f"{pts[3]:.1f}",
    })

print(f"Model parameters: γ={GAMMA} segments, δ={DELTA}, "
      f"weight bounds=[{WEIGHT_LB}, {WEIGHT_UB}], "
      f"min segment share={MIN_SEGMENT_SHARE}, "
      f"non-linearity threshold={NON_LINEARITY_THRESHOLD}")
print()
pd.DataFrame(cp_data)

Model parameters: γ=3 segments, δ=0.001, weight bounds=[0.0625, 0.5], min segment share=0.05, non-linearity threshold=0.1



,Criterion,Nature,x⁰ (worst),x¹,x²,x³ (best)
0,Employment rate,gain,56.0,64.0,72.0,80.0
1,Long-term unemployment rate,cost,10.8,7.4,4.0,0.6
2,Personal earnings,gain,23619.0,38242.0,52865.0,67488.0
3,Life expectancy,gain,75.5,78.3,81.2,84.0
4,Life satisfaction,gain,5.8,6.5,7.2,7.9
5,Employees working very long hours,cost,11.7,7.9,4.1,0.3
6,Air pollution,cost,22.8,17.0,11.3,5.5
7,Distance from Poznan (km),cost,2562.8,1708.5,854.3,0.0


## 4. Inconsistency Resolution — Execution

We iteratively solve the MILP to find **all minimal removal sets**. After finding each set, we add a cut preventing the solver from finding the same solution again. The process stops when the MILP becomes infeasible.

In [16]:
removals = find_all_minimal_removals(df, directions, preferences)

Iteration 1: V* = 1, removed = [7]
Iteration 2: V* = 1, removed = [9]
Iteration 3: V* = 1, removed = [8]

Iteration 4: Infeasible — all minimal removal sets found.


## 5. Results

The solver found **3 minimal removal sets**, each of size 1. This means only **one comparison** needs to be removed to restore consistency — and there are exactly 3 ways to do it, each corresponding to removing one edge of the cycle Denmark ≻ Sweden ≻ Norway ≻ Denmark.

In [17]:
# Display all removal sets as a summary table
results = []
for idx, removal_set in enumerate(removals):
    removed = [f"{preferences[k][0]} ≻ {preferences[k][1]}" for k in sorted(removal_set)]
    kept = [f"{preferences[k][0]} ≻ {preferences[k][1]}" for k in range(len(preferences)) if k not in removal_set]
    results.append({
        "Removal set": idx + 1,
        "Removed comparison": ", ".join(removed),
        "Consistent subset size": len(preferences) - len(removal_set),
    })

pd.DataFrame(results)

,Removal set,Removed comparison,Consistent subset size
0,1,Denmark ≻ Sweden,9
1,2,Norway ≻ Denmark,9
2,3,Sweden ≻ Norway,9


In [18]:
# Detailed view of each consistent subset
for idx, removal_set in enumerate(removals):
    print(f"{'='*60}")
    print(f"Consistent subset {idx + 1} (removed: {[preferences[k] for k in removal_set]})")
    print(f"{'='*60}")
    for k in range(len(preferences)):
        pref, over = preferences[k]
        status = "REMOVED" if k in removal_set else "kept"
        marker = "  ✗ " if k in removal_set else "  ✓ "
        print(f"{marker}[{k}] {pref} ≻ {over}")
    print()

Consistent subset 1 (removed: [('Denmark', 'Sweden')])
  ✓ [0] Switzerland ≻ Netherlands
  ✓ [1] Germany ≻ Czech Republic
  ✓ [2] Norway ≻ United Kingdom
  ✓ [3] Finland ≻ Poland
  ✓ [4] Netherlands ≻ Belgium
  ✓ [5] Luxembourg ≻ Ireland
  ✓ [6] Germany ≻ Slovenia
  ✗ [7] Denmark ≻ Sweden
  ✓ [8] Sweden ≻ Norway
  ✓ [9] Norway ≻ Denmark

Consistent subset 2 (removed: [('Norway', 'Denmark')])
  ✓ [0] Switzerland ≻ Netherlands
  ✓ [1] Germany ≻ Czech Republic
  ✓ [2] Norway ≻ United Kingdom
  ✓ [3] Finland ≻ Poland
  ✓ [4] Netherlands ≻ Belgium
  ✓ [5] Luxembourg ≻ Ireland
  ✓ [6] Germany ≻ Slovenia
  ✓ [7] Denmark ≻ Sweden
  ✓ [8] Sweden ≻ Norway
  ✗ [9] Norway ≻ Denmark

Consistent subset 3 (removed: [('Sweden', 'Norway')])
  ✓ [0] Switzerland ≻ Netherlands
  ✓ [1] Germany ≻ Czech Republic
  ✓ [2] Norway ≻ United Kingdom
  ✓ [3] Finland ≻ Poland
  ✓ [4] Netherlands ≻ Belgium
  ✓ [5] Luxembourg ≻ Ireland
  ✓ [6] Germany ≻ Slovenia
  ✓ [7] Denmark ≻ Sweden
  ✗ [8] Sweden ≻ Norway
  ✓ [9]

## 6. Discussion — Choosing the Consistent Subset

All 7 non-cycle comparisons appear in every consistent subset — the inconsistency is entirely localized in the Scandinavian cycle (Denmark ≻ Sweden ≻ Norway ≻ Denmark). The solver found 3 minimal removal sets, each removing exactly one edge of this cycle:

| Subset | Removed comparison | Implied Scandinavian ranking |
|--------|-------------------|------------------------------|
| 1 | Denmark ≻ Sweden [7] | Sweden ≻ Norway ≻ Denmark |
| 2 | Norway ≻ Denmark [9] | Denmark ≻ Sweden ≻ Norway |
| 3 | Sweden ≻ Norway [8] | Norway ≻ Denmark ≻ Sweden |

### Decision: Remove Sweden ≻ Norway (Subset 3)

We choose to remove **Sweden ≻ Norway** (comparison [8]) — the comparison the DM is least confident about:

- **Denmark ≻ Sweden** is well-justified: Denmark offers significantly higher earnings and is much closer to Poznan, which are key factors for a student planning to relocate.
- **Norway ≻ Denmark** is strongly supported: Norway excels in air quality, life expectancy, and employment rate — criteria that directly impact quality of life.
- **Sweden ≻ Norway**, on the other hand, rests on marginal advantages in pollution levels and work-life balance, which are less decisive compared to Norway's broader strengths.

The resulting consistent subset retains **9 out of 10** original comparisons and implies the transitive Scandinavian ranking **Norway ≻ Denmark ≻ Sweden**. This reflects a decision maker who prioritizes quality-of-life factors (health, environment, employment) while also valuing economic opportunity and proximity to home.